<a href="https://colab.research.google.com/github/shemo203/calmrocks-personal/blob/main/00-setup/00-environment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environment & Cost Hygiene

**Goal:** Set up API keys via Colab secrets, add spend guards, make one successful model call, and feel why LLM systems can't be tested like normal code.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).


In [77]:
%pip install tiktoken

In [78]:
%pip install -q groq

## API keys via Colab Secrets

Never paste an API key into a notebook cell. Notebooks get shared, committed, and screenshotted, and a key in a cell is a key you'll be rotating next week.

Get your free Groq API key at **console.groq.com** (no credit card required).

Then add it to Colab:

1. Click the **key icon** in the left sidebar ("Secrets").
2. Click **Add new secret**. Name it exactly `GROQ_API_KEY`, paste your key as the value.
3. Flip the **Notebook access** toggle on for this notebook.

The cell below reads that secret in Colab, or falls back to the `GROQ_API_KEY` environment variable if you're running locally.


In [79]:
import os

# Load GROQ_API_KEY from Colab Secrets (key icon, left sidebar) or, if you're
# running locally, from the GROQ_API_KEY environment variable.
try:
    from google.colab import userdata  # only importable inside Colab
    try:
        os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
    except Exception as e:
        # In Colab, but the secret is missing or notebook access is off.
        raise RuntimeError(
            "Couldn't read GROQ_API_KEY from Colab Secrets. Click the key icon "
            "in the left sidebar, add a secret named exactly GROQ_API_KEY, and "
            "flip 'Notebook access' on for this notebook, then re-run this cell."
        ) from e
except ImportError:
    # Not in Colab — expect the key in the environment.
    if not os.environ.get('GROQ_API_KEY'):
        raise RuntimeError(
            'GROQ_API_KEY is not set. Run `export GROQ_API_KEY=...` in your '
            'shell before launching Jupyter, or set it in os.environ.'
        )

from groq import Groq

# max_retries=8 (SDK default is 2): the free tier caps tokens-per-minute low, so a
# burst of calls can hit HTTP 429. The SDK retries 429s with backoff and honors the
# server's retry-after hint, so a higher ceiling just waits out those short windows
# transparently instead of erroring. Costs nothing when you're under the limit.
client = Groq(max_retries=8)
MODEL = 'openai/gpt-oss-120b'  # update if you get a 404 — run the cells below to see what's available
print('Groq client ready. Default MODEL:', MODEL)

Groq client ready. Default MODEL: openai/gpt-oss-120b


## Listing available models

Groq's catalog changes over time: models get added, renamed, and retired. So don't hardcode a model name and forget it. If a call fails with `404 / model_not_found`, run the cell below to see what's live right now, then update `MODEL` above.

The catalog also lists non-chat models (speech-to-text, text-to-speech, safety/guard models). The cell filters those out so you're choosing from chat models only.

**How to pick one:**
- **Default to the largest chat model** (highest parameter count in the name, so `120b` beats `20b`). It's strong enough for almost everything while you're iterating.
- **Drop to a smaller model for bulk, repetitive work.** Classification, extraction over many rows, judge calls inside eval loops, anything you run N times where speed and cost multiply.

In [80]:
# List the chat models currently available on your Groq account.
# If MODEL above gives a 404, pick a replacement from this list and update it.

# Substrings that mark non-chat models (speech-to-text, text-to-speech, safety).
NON_CHAT = ('whisper', 'tts', 'guard', 'embed', 'distil-whisper')

models = client.models.list()
chat_models = sorted(
    m.id for m in models.data
    if not any(tag in m.id.lower() for tag in NON_CHAT)
)

print(f'{len(chat_models)} chat models available:\n')
for m in chat_models:
    marker = '  <- current MODEL' if m == MODEL else ''
    print(f'  {m}{marker}')

if MODEL not in chat_models:
    print(f'\n⚠️  Current MODEL ({MODEL!r}) is not in the list above — pick one and update the MODEL cell.')

9 chat models available:

  allam-2-7b
  canopylabs/orpheus-arabic-saudi
  canopylabs/orpheus-v1-english
  groq/compound
  groq/compound-mini
  openai/gpt-oss-120b  <- current MODEL
  openai/gpt-oss-20b
  qwen/qwen3.6-27b
  qwen/qwen3.8-27b


## First call

One request, one response. Groq follows the OpenAI-compatible format: `response.choices[0].message.content` is the text, and `response.usage` is the billing record for this call.


In [81]:
response = client.chat.completions.create(
    model=MODEL,
    max_tokens=200,
    messages=[{'role': 'user', 'content': 'In two sentences: what does a forward-deployed engineer do?'}],
)

print(response.choices[0].message.content)
print()
print('finish_reason:', response.choices[0].finish_reason)
print('usage:', response.usage)


A forward‑deployed engineer works directly with customers to integrate, customize, and optimize a company’s technology solutions on‑site, translating client needs into functional implementations and rapid prototypes. They also serve as technical consultants, troubleshooting issues, gathering feedback, and helping shape product roadmaps based on real‑world usage.

finish_reason: stop
usage: CompletionUsage(completion_tokens=105, prompt_tokens=84, total_tokens=189, completion_time=0.221279673, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=34), prompt_time=0.003581243, prompt_tokens_details=None, queue_time=0.384304096, total_time=0.224860916)


## The shock: this is not normal software

Before anything else, feel the one thing that makes building on LLMs different from every backend system you've shipped. Run the cell below (the **same request, three times**) and watch the answers *not* match. Then watch a strict output contract break intermittently.

In normal software, same input → same output, and a test that passes once passes always. Here, "it worked when I tried it" guarantees nothing about the next call. That single fact is why this whole course is organized around **measuring** behavior instead of eyeballing it, and why section 02 installs evals before you build anything you'd need to trust.

In [82]:
# Same prompt, three times. In normal code this would be three identical results.
prompt = 'Name one benefit of RAG. Answer in exactly 3 words.'
print('--- same prompt x3 (temperature=1) ---')
for i in range(3):
    r = client.chat.completions.create(
        model=MODEL, max_tokens=20, temperature=1,
        messages=[{'role': 'user', 'content': prompt}],
    )
    print(f'  {i+1}: {r.choices[0].message.content.strip()!r}')

# Now a strict output *contract*: "only a number." Run several times and watch it
# occasionally add words, punctuation, or a sentence — the kind of drift that
# silently breaks `int(response)` downstream in production.
print('\n--- strict contract: "reply with ONLY the number" x3 ---')
for i in range(3):
    r = client.chat.completions.create(
        model=MODEL, max_tokens=20, temperature=1,
        messages=[{'role': 'user',
                   'content': 'How many bytes in a kilobyte? Reply with ONLY the number, nothing else.'}],
    )
    out = r.choices[0].message.content.strip()
    parses = out.isdigit()
    print(f'  {i+1}: {out!r}   int()-parseable? {parses}')

--- same prompt x3 (temperature=1) ---
  1: ''
  2: ''
  3: ''

--- strict contract: "reply with ONLY the number" x3 ---
  1: ''   int()-parseable? False
  2: ''   int()-parseable? False
  3: ''   int()-parseable? False


## Reading `response.usage`

Every response carries exact token accounting. This is your ground truth for cost:

- `prompt_tokens`: input tokens (everything you sent: system, history, this message)
- `completion_tokens`: output tokens (what the model generated)
- `total_tokens`: the sum

Groq has no prompt caching, so there are no cache fields to reason about. Cost is just each bucket times its per-token rate, which is all the spend guard below needs to do its job.

In [88]:
import time  # used by the exercises at the end
import tiktoken
from statistics import median

PRICES = {
    # USD per million tokens: (input, output). The Groq free tier costs nothing —
    # these are here so the cost math is real for comparison exercises and for when
    # you move to a paid tier. Always check console.groq.com/pricing for current rates.
    'openai/gpt-oss-120b': (0.90, 0.90),
    'openai/gpt-oss-20b':  (0.15, 0.15),
}
DEFAULT_RATE = (0.90, 0.90)  # fallback for models not in PRICES


class BudgetExceeded(Exception):
    """Raised when a call would push session spend past the budget."""


class SpendGuard:
    """Drop-in wrapper around client.chat.completions.create.

    Tracks estimated cost per call, keeps a running session total, and refuses
    to make a call once the budget is exhausted. Same signature as the wrapped
    method, so you can swap `client.chat.completions.create(...)` for
    `guard.create(...)` anywhere.
    """

    def __init__(self, budget_usd=0.50, hard_max_tokens = 0):
        self.budget = budget_usd
        self.spent = 0.0
        self.calls = 0
        self.values = []

    def _cost(self, model, usage):
        inp_rate, out_rate = PRICES.get(model, DEFAULT_RATE)
        return (usage.prompt_tokens * inp_rate + usage.completion_tokens * out_rate) / 1_000_000

    def create(self, **kwargs):
        if self.spent >= self.budget:
            raise BudgetExceeded(
                f'Session spend ${self.spent:.4f} >= budget ${self.budget:.2f} — refusing to call.'
            )
        token_count = 0
        encoder = tiktoken.get_encoding("cl100k_base")
        for message in kwargs["messages"]:
          token_count += len(encoder.encode(message["content"])) # Corrected line
        if token_count >= kwargs["hard_max_tokens"]:
          raise Exception("Rejected, exceeds max token count")
        start = time.monotonic()
        response = client.chat.completions.create(**kwargs)
        end = time.monotonic()
        self.values.append(end - start)
        cost = self._cost(kwargs['model'], response.usage)
        self.spent += cost
        self.calls += 1
        print(
            f'[spend_guard] call {self.calls}: '
            f'{response.usage.prompt_tokens} in / {response.usage.completion_tokens} out '
            f'~= ${cost:.5f}  (session: ${self.spent:.5f} / ${self.budget:.2f})'
        )
        return response

    def report(self):
        print(f"Total spent : {self.spent}")
        print(f"Total calls: {self.calls}")
        if self.values:
          print(f"max latency: {max(self.values)}")
          print(f"p50: {median(self.values)}")
        else:
          print("No recorded latencies")



guard = SpendGuard(budget_usd=0.25)
print(f'SpendGuard ready — budget ${guard.budget:.2f}.')

SpendGuard ready — budget $0.25.


In [89]:
# Normal use: same signature as client.chat.completions.create, plus a running tally.
resp = guard.create(
    model=MODEL,
    max_tokens=100,
    messages=[{'role': 'user', 'content': 'One sentence: why do teams add spend guards around LLM calls?'}],
    hard_max_tokens = 10

)
print(resp.choices[0].message.content)


Exception: Rejected, exceeds max token count

In [ ]:
# And the failure path: a deliberately tiny budget trips after the first call.
tiny = SpendGuard(budget_usd=0.0001)
try:
    tiny.create(model=MODEL, max_tokens=50,
                messages=[{'role': 'user', 'content': 'Say hi.'}])  # this one succeeds...
    tiny.create(model=MODEL, max_tokens=50,
                messages=[{'role': 'user', 'content': 'Say hi again.'}])  # ...this one raises
except BudgetExceeded as e:
    print('Guard tripped as expected:', e)


In [ ]:
print(guard.report())

## Picking a model (on Groq)

The names will change; the principle won't. Two main tiers available free on Groq:

- **120B-class** (`openai/gpt-oss-120b`): the iterate-on default. Strong enough for almost everything.
- **20B-class** (`openai/gpt-oss-20b`): bulk and cheap operations. Classification, extraction over thousands of rows, judge calls in eval loops, anything where you multiply by N.

The working rule: **develop on cheap, eval on target.** Build the pipeline against the 20B model, then run your evaluation set against the 120B model you'll actually ship before you commit. Prompt behavior shifts between tiers (a prompt tuned on 20B may behave differently on 120B), so the eval-on-target step is not optional.

In [ ]:
# Same prompt on two tiers — compare cost and feel. (2 API calls.)
prompt = 'Classify the sentiment of this review as positive, negative, or mixed: '\
         '"The battery life is great but the screen scratched in a week." Reply with one word.'

for model in ['openai/gpt-oss-20b', 'openai/gpt-oss-120b']:
    r = guard.create(model=model, max_tokens=10,
                     messages=[{'role': 'user', 'content': prompt}])
    print(f'{model}: {r.choices[0].message.content.strip()}')

In [ ]:

for prompt in ['Explain the different economic and millitary situations of the seven kingdoms in the Game of Thrones', '* for each kingdoms in AGOT * millitary * economic status']:
    r = guard.create(model=model, max_tokens=1,
                     messages=[{'role': 'user', 'content': prompt}])


Run it and note that both tiers nail a task this simple, which is exactly the point. If the smaller model passes your eval for a task, routing it to the larger one is just burning margin.

## Exercises

1. Extend `SpendGuard.create` to record per-call wall-clock latency (`time.monotonic()` around the request) and add a `.report()` method that prints total cost, total calls, and p50/max latency.
2. Take a prompt you care about and phrase it two ways: terse bullet-point style vs. full prose. Compare `prompt_tokens` for each (send both with `max_tokens=1` to keep it cheap) and compute the cost difference at 100k calls/month.
3. Add a `hard_max_tokens` option to `SpendGuard` that rejects any call requesting more than N output tokens, so a typo like `max_tokens=100000` can't slip through.
4. The free Groq tier has rate limits (requests/minute and tokens/minute). Add a retry loop to `SpendGuard.create` that catches `groq.RateLimitError` and backs off with `time.sleep`.